# Step 1: data and linear probes

One-time setup:
1. Make a folder `grsats` in My Drive and upload `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip`, and `finalfitz17k.zip` (the `finalfitz17k` folder zipped) into it.
2. Make a GitHub token with read access to the repo and add it in Colab under Secrets (key icon) as `GH_TOKEN`.
3. Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo
%cd /content/repo/code
!pip install -q -r requirements.txt

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

In [ ]:
# unzip images to the colab disk, reading thousands of files straight from drive is really slow
# (same folders you get on a mac by double clicking the zips)
ham = "../data/HAM10000/dataverse_files"
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python -m scripts.check_data
!python -m pytest -q tests

In [ ]:
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.extract_features --dataset {ds} --backbone {bb}

In [ ]:
# cpu only, could also run this locally after downloading the outputs folder
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.train_probes --dataset {ds} --backbone {bb}